# SupplyGuard Saudi — EDA notebook

This notebook walks through the raw data, applies preprocessing, computes features and the composite risk score, then produces the EDA visuals saved into `data/outputs/figures/`.

Run from the project root:

```bash
jupyter notebook notebooks/01_eda_supplyguard.ipynb
```

Or simply execute `python run_pipeline.py` — this notebook mirrors the pipeline for interactive exploration.

In [ ]:
import sys, os, warnings
sys.path.insert(0, os.path.abspath('..'))
warnings.filterwarnings('ignore')

import pandas as pd, numpy as np
import matplotlib.pyplot as plt

from src import data_loader, preprocessing, feature_engineering, risk_scoring, anomaly_detection, evaluation, visualization, eda, config

## 1. Load raw data

In [ ]:
main = data_loader.load_main()
events = data_loader.load_events()
pw_wide = data_loader.load_portwatch_wide()
pw_long = data_loader.load_portwatch_long()
methodology = data_loader.load_methodology()
for name, df in [('main', main), ('events', events), ('pw_wide', pw_wide), ('pw_long', pw_long)]:
    data_loader.describe_frame(df, name)

## 2. Preprocessing

In [ ]:
cleaned = preprocessing.clean_main(main)
cleaned.head()

## 3. Feature engineering

In [ ]:
feats = feature_engineering.build_all_features(cleaned, events)
feats.shape, feats.columns.tolist()[:30]

## 4. Composite risk scoring

In [ ]:
scored = risk_scoring.compute_supplyguard_score(feats)
scored['supplyguard_risk_score_0_100'].describe()

## 5. Anomaly detection

In [ ]:
scored = anomaly_detection.run_anomaly_pipeline(scored)
scored[scored.anomaly_flag == 1].sort_values('anomaly_score', ascending=False).head(10)

## 6. Visualisations

In [ ]:
visualization.generate_all_figures(scored, events)
eda.print_summary(scored, events)

## 7. Event backtesting

In [ ]:
evaluation.run_event_backtest(scored, events)